# 04. Ejecutar experimentos

Corre el batch de combinaciones (modelo x tecnica de balanceo x metodo de optimizacion) via `experiment_runner.run_experiments()`, con registro **incremental** en `results/experiments_master.csv`: cada combinacion se guarda apenas termina, y una corrida posterior con `resume=True` (default) omite las que ya estan en el CSV. Se puede interrumpir y continuar despues, o correr un subconjunto ahora (ej. modelos baratos en local) y el resto despues (ej. SVM/KNN en Colab).

**Datos:** el dataset real ya viene preprocesado desde el EDA (`00_eda.ipynb`, Parte 5): sin duplicados, con las categorias no oficiales reagrupadas, variables monetarias en log y el bloque `BILL_AMT` resumido. `TARGET_COLUMN`/`CATEGORICAL_COLUMNS` en `config.py` ya estan confirmados. La celda de carga igual vuelve a verificar duplicados (deberian ser 0) como red de seguridad.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np

from src import config
from src.data_loader import load_processed_dataset, split_features_target
from src.synthetic_data import generate_synthetic_dataset
from src.preprocessing import remove_duplicate_rows, check_label_conflicts
from src.models import MODEL_REGISTRY
from src.experiment_runner import run_experiments, get_all_combinations, MODEL_COST_ORDER
from src.evaluation import format_metrics_table

np.random.seed(config.RANDOM_STATE)

## Cargar datos

Usa el dataset real preprocesado en `data/processed/` (ver `config.PROCESSED_DATA_PATH`); si por alguna razon no existe, cae a datos sinteticos con la misma forma (util para probar el pipeline).

In [2]:
try:
    df = load_processed_dataset()
    print(f"Dataset real cargado: {df.shape}")
except FileNotFoundError as e:
    print("Dataset real aun no disponible, usando datos sinteticos para desarrollo/pruebas:")
    print(f"  {e}")
    df = generate_synthetic_dataset()
    print(f"Dataset sintetico generado: {df.shape}")

df, n_removed = remove_duplicate_rows(df)
print(f"Filas duplicadas eliminadas: {n_removed}")

n_conflicts = check_label_conflicts(df)
if n_conflicts > 0:
    print(
        f"AVISO: {n_conflicts} grupos de filas con features identicas pero target distinto "
        "(ruido irreducible del dataset, no se resuelve automaticamente)."
    )

X, y = split_features_target(df)
print(f"X: {X.shape}, balance del target: {y.value_counts(normalize=True).round(4).to_dict()}")

Dataset real aun no disponible, usando datos sinteticos para desarrollo/pruebas:
  No se encontró el dataset procesado en C:\Users\valef\OneDrive\Desktop\Credit_Card_Project\data\processed\dataset_final.csv. Usa src.synthetic_data.generate_synthetic_dataset() mientras llega el dataset final, o ajusta config.PROCESSED_DATA_PATH.
Dataset sintetico generado: (5000, 21)
Filas duplicadas eliminadas: 0
X: (5000, 20), balance del target: {0: 0.7754, 1: 0.2246}


## Configuración de la corrida

Corre la grilla completa: 7 modelos base × 4 técnicas de balanceo × 4 métodos de optimización, con los presupuestos reales definidos en `config.py` (`N_OUTER_FOLDS`, `N_INNER_FOLDS`, `RANDOM_SEARCH_ITER`, `OPTUNA_N_TRIALS`, `GA_POPULATION_SIZE`/`GA_N_GENERATIONS`). `get_all_combinations` filtra automáticamente las combinaciones (modelo, técnica) incompatibles: KNN y Naive Bayes no aceptan `class_weight` en scikit-learn (`KNeighborsClassifier` y `GaussianNB` no tienen ese parámetro), así que de las 112 combinaciones teóricas quedan **104 válidas**; las 8 restantes (KNN/Naive Bayes × class_weight) se documentan como no aplicables en `05_results_analysis.ipynb` en vez de forzarse con una alternativa no equivalente. Ver `experiment_runner.MODEL_COST_ORDER` para el orden de ejecución (baratos primero).

In [ ]:
models = None  # los 7 modelos base (los otros 6 ya estan 100% completos, resume=True los salta)

# Reordenado (2026-09-23, ~presupuesto de 6h para las 12 combinaciones de
# svm que faltan): class_weight primero (mas barato, no agranda el dataset
# como smote/adasyn -- costo similar a la tecnica "none" ya medida:
# grid=57min, random=64min, bayesian=65min), despues smote, despues adasyn.
# methods excluye genetic_deap por ahora: es el metodo mas caro por lejos
# (svm/none/genetic_deap tardo 4h21min el solo) -- se corre aparte despues,
# si sobra tiempo, para no arriesgar que consuma todo el presupuesto y deje
# 0 combinaciones nuevas completas. Ver EXPERIMENT_STATUS.md seccion 8q.
techniques = ["class_weight", "smote", "adasyn"]
methods = ["grid_search", "random_search", "bayesian_optuna"]
n_outer_folds, n_inner_folds = config.N_OUTER_FOLDS, config.N_INNER_FOLDS

# n_jobs por metodo (ver config.OPTIMIZATION_N_JOBS): paraleliza el eje de
# evaluaciones independientes de CADA metodo (candidatos de grid/random
# search, trials de Optuna, individuos de la poblacion en genetic_deap).
# No cambia folds/rangos/tamanio de poblacion/iteraciones, solo el tiempo de
# pared -- ver EXPERIMENT_STATUS.md seccion "Paralelismo (2026-09-22)".
optimizer_kwargs_by_method = {
    "grid_search": {"n_jobs": config.OPTIMIZATION_N_JOBS},
    "random_search": {"n_iter": config.RANDOM_SEARCH_ITER, "n_jobs": config.OPTIMIZATION_N_JOBS},
    "bayesian_optuna": {"n_trials": config.OPTUNA_N_TRIALS, "n_jobs": config.OPTIMIZATION_N_JOBS},
    "genetic_deap": {
        "population_size": config.GA_POPULATION_SIZE,
        "n_generations": config.GA_N_GENERATIONS,
        "n_jobs": config.OPTIMIZATION_N_JOBS,
    },
}
# n_jobs=1 aqui (bucle EXTERNO sobre combinaciones, secuencial): el
# paralelismo real ahora vive dentro de cada metodo (ver arriba), para no
# anidar paralelismo (evita sobre-suscribir los nucleos disponibles).
n_jobs = 1

# timeout_seconds=None: DESACTIVADO (ver EXPERIMENT_STATUS.md seccion 8d).
# El wrapper de multiprocessing.Process para el tope de tiempo anida
# paralelismo de procesos con el n_jobs de arriba y se colgo en la corrida
# real. El backstop para combinaciones atipicamente lentas se maneja por
# fuera (monitor externo), no aqui.
timeout_seconds = None

combos = get_all_combinations(models, techniques, methods)
print(f"{len(combos)} combinaciones a considerar.")
print(f"Orden por costo computacional: {MODEL_COST_ORDER}")

## Ejecutar (incremental, reanudable)

`resume=True` (default) salta las combinaciones que ya esten en `results/experiments_master.csv`; se puede volver a correr esta celda tantas veces como haga falta (ej. tras agregar mas modelos a la lista de arriba) sin repetir trabajo ya hecho.

In [ ]:
table, results = run_experiments(
    X,
    y,
    models=models,
    techniques=techniques,
    methods=methods,
    n_outer_folds=n_outer_folds,
    n_inner_folds=n_inner_folds,
    optimizer_kwargs_by_method=optimizer_kwargs_by_method,
    checkpoint_path=config.EXPERIMENTS_MASTER_PATH,
    resume=True,
    n_jobs=n_jobs,
    timeout_seconds=timeout_seconds,
)

print(f"\n{len(results)} combinaciones nuevas corridas en esta sesion.")
print(f"{len(table)} filas totales en la tabla maestra ({config.EXPERIMENTS_MASTER_PATH}).")

## Resultado

Tabla de métricas (media ± desviación estándar del outer loop, nunca un valor puntual), ordenada por la métrica principal de validación (ver `config.DEFAULT_SCORING` y la justificación en `evaluation.py`).

In [5]:
metric_for_ranking = config.DEFAULT_SCORING
table_sorted = table.sort_values(f"{metric_for_ranking}_mean", ascending=False)
format_metrics_table(table_sorted)

,model,technique,method,accuracy,precision,recall,f1,roc_auc
3,decision_tree,none,random_search,0.8552 ± 0.0121,0.6894 ± 0.0213,0.6447 ± 0.0444,0.6661 ± 0.0335,0.8513 ± 0.0117
4,decision_tree,class_weight,random_search,0.8194 ± 0.0063,0.5706 ± 0.0098,0.7916 ± 0.0184,0.6632 ± 0.0126,0.8647 ± 0.0131
1,logistic_regression,none,random_search,0.8628 ± 0.0098,0.7685 ± 0.027,0.5565 ± 0.03,0.6454 ± 0.0285,0.8609 ± 0.0067
2,logistic_regression,class_weight,random_search,0.7886 ± 0.0099,0.52 ± 0.0154,0.789 ± 0.0081,0.6266 ± 0.0086,0.8623 ± 0.0029
0,naive_bayes,none,random_search,0.7982 ± 0.0056,0.5433 ± 0.0138,0.65 ± 0.0287,0.5912 ± 0.0045,0.8113 ± 0.0058


Continuar en `05_results_analysis.ipynb`: funciona directamente desde `results/experiments_master.csv`, en una sesión nueva, sin necesitar nada de este notebook en memoria.